### 04_silver_to_gold_ingestion

Connect to Postgre > Create the tables for Gold layer > Load data from Silver into the tables > Validate it

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Parameters

In [2]:
silver_dir = f"{project_root}/data/silver"
truncate_before_load = True
validation_limit = 5

## Imports

In [3]:
import pandas as pd

from src.db.postgres_client import PostgresClient
from src.db.create_tables import PostgresSchemaCreator
from src.ingestion.silver_to_gold_loader import SilverToGoldIngestion

## Test connection

In [4]:
client = PostgresClient()
client.test_connection()

Connection to PostgreSQL established successfully.


## Create Gold tables

In [5]:
creator = PostgresSchemaCreator()
creator.run()

Connection to PostgreSQL established successfully.
Creating table: dim_tempo
Table created successfully: dim_tempo
Creating table: dim_localidade
Table created successfully: dim_localidade
Creating table: dim_condicao_via
Table created successfully: dim_condicao_via
Creating table: dim_vitima
Table created successfully: dim_vitima
Creating table: dim_tipo_veiculo
Table created successfully: dim_tipo_veiculo
Creating table: fato_acidente
Table created successfully: fato_acidente
Creating table: fato_vitima
Table created successfully: fato_vitima
Creating table: fato_veiculo_acidente
Table created successfully: fato_veiculo_acidente
All PostgreSQL tables were created successfully.


## Build Gold tables from Silver and load into PostgreSQL

In [6]:
loader = SilverToGoldIngestion(silver_dir=str(silver_dir))
loader.run(truncate_before_load=truncate_before_load)

Connection to PostgreSQL established successfully.
Running in FACTS-ONLY mode
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0001.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0002.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0003.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0004.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0005.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0006.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0007.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0008.parquet
Truncating FACT tables only...
Loa

ValueError: Null foreign keys detected in fato_veiculo_acidente: {'pk_tempo': 12, 'pk_localidade': 12}

## Validation queries

In [7]:
validation_tables = [
    "dim_tempo",
    "dim_localidade",
    "dim_condicao_via",
    "dim_vitima",
    "dim_tipo_veiculo",
    "fato_acidente",
    "fato_vitima",
    "fato_veiculo_acidente",
]

for table_name in validation_tables:
    print(f"\n--- {table_name} ---")
    display(loader.validate_table(table_name=table_name, limit=validation_limit))


--- dim_tempo ---


,pk_tempo,data,ano,mes,mes_ano,dia_semana
0,1,2018-01-01,2018,1,201801,Monday
1,2,2018-01-02,2018,1,201801,Tuesday
2,3,2018-01-03,2018,1,201801,Wednesday
3,4,2018-01-04,2018,1,201801,Thursday
4,5,2018-01-05,2018,1,201801,Friday



--- dim_localidade ---


,pk_localidade,localidade,ano_ref,mes_ref,regiao,uf,cod_ibge,municipio,regiao_metropolitana,qtde_habitantes,frota_total,frota_circulante
0,1,AC0000000201801,2018,1,NORTE,AC,0,NAO INFORMADO,nao,0,0,0
1,2,AC0000000201802,2018,2,NORTE,AC,0,NAO INFORMADO,nao,0,0,0
2,3,AC0000000201803,2018,3,NORTE,AC,0,NAO INFORMADO,nao,0,0,0
3,4,AC0000000201804,2018,4,NORTE,AC,0,NAO INFORMADO,nao,0,0,0
4,5,AC0000000201805,2018,5,NORTE,AC,0,NAO INFORMADO,nao,0,0,0



--- dim_condicao_via ---


...

,pk_fato_acidente,num_acidente,pk_tempo,pk_localidade,pk_condicao_via,qtde_acidente,qtde_acid_com_obitos,qtde_envolvidos,qtde_feridosilesos,qtde_obitos
0,1,1068969,8,6531,851,1,0,1,1,0
1,2,1068969,8,6532,851,1,0,1,1,0
2,3,1068969,8,6533,851,1,0,1,1,0
3,4,1068969,8,6534,851,1,0,1,1,0
4,5,1068969,8,6535,851,1,0,1,1,0



--- fato_vitima ---


,pk_fato_vitima,num_acidente,pk_tempo,pk_localidade,pk_vitima,flag_obito,flag_ferido
0,1,3830599,11,5163,2466,False,False
1,2,3830599,11,5164,2466,False,False
2,3,3830599,11,5165,2466,False,False
3,4,3830599,11,5166,2466,False,False
4,5,3830599,11,5167,2466,False,False



--- fato_veiculo_acidente ---


,pk_fato_veiculo,num_acidente,pk_tempo,pk_localidade,pk_tipo_veiculo,qtde_veiculos
0,1,14,1051,356598,63,1
1,2,14,1051,356599,63,1
2,3,14,1051,356600,63,1
3,4,14,1051,356601,63,1
4,5,14,1051,356602,63,1


## Optional row count check

In [8]:
for table_name in validation_tables:
    query = f"SELECT COUNT(*) AS total_rows FROM {client.schema}.{table_name}"
    with client.get_connection() as conn:
        count_df = pd.read_sql(query, conn)
    print(f"{table_name}: {int(count_df.loc[0, 'total_rows'])}")

dim_tempo: 2435
dim_localidade: 2047244
dim_condicao_via: 1206
dim_vitima: 9848
dim_tipo_veiculo: 105
fato_acidente: 218938663
fato_vitima: 345076255
fato_veiculo_acidente: 160765385
